# Reducción de Dimensionalidad — PCA _(versión detallada)_

_Componentes principales, varianza explicada y visualización de clusters K-Means + PCA_

**Módulo 2 — Aprendizaje No Supervisado** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

---
> 📖 **Nota:** Esta es la versión detallada del notebook `03_reduccion_dimensionalidad_pca.ipynb`.  
> Incluye teoría ampliada y comentarios línea a línea en cada bloque de código.

![Aprendizaje No Supervisado](assets/header.png)

## 1. ¿Por qué reducir dimensionalidad?

Cuando $p$ (número de features) es grande (decenas, cientos o miles de variables), aparecen múltiples problemas que afectan el rendimiento y la interpretabilidad de los modelos.

### 1.1 La maldición de la dimensión (Curse of Dimensionality)

**Problema:** Al aumentar $p$, el volumen del espacio crece **exponencialmente**, pero el número de observaciones $n$ crece **linealmente** (o se mantiene fijo).

**Consecuencias:**

1. **Datos esparsos:** Los puntos están muy dispersos en el espacio
   - Con $p=2$ y 1000 puntos: densidad razonable
   - Con $p=100$ y 1000 puntos: casi vacío (1000 puntos en un espacio de 10¹⁰⁰ dimensiones)

2. **Distancias se vuelven similares:** 
   - En alta dimensión, la distancia entre el punto más cercano y el más lejano tiende a ser similar
   - Los algoritmos basados en distancia (K-Means, KNN) pierden poder discriminativo

3. **Overfitting:**
   - Con muchas features, los modelos pueden "memorizar" ruido
   - Necesitas exponencialmente más datos para mantener la misma densidad

**Ejemplo numérico:**

Para mantener la misma densidad de datos:
- Con $p=2$: necesitas 100 puntos para cubrir el espacio
- Con $p=10$: necesitas 100⁵ = 10 mil millones de puntos
- Con $p=100$: necesitas 100⁵⁰ puntos (más que átomos en el universo)

### 1.2 Redundancia y correlación

En datasets reales, muchas variables suelen estar **correlacionadas** y aportan información **repetida**.

**Ejemplos:**
- En datos de clientes: `TotalCharges` ≈ `MonthlyCharges` × `tenure`
- En imágenes: píxeles vecinos suelen tener valores similares
- En finanzas: precios de acciones del mismo sector están correlacionados

**Implicación:** Podemos representar los datos con menos dimensiones sin perder mucha información.

### 1.3 Visualización

Los humanos solo podemos visualizar 2-3 dimensiones. Con $p > 3$, necesitamos proyectar a 2D o 3D para:
- Explorar la estructura de los datos
- Detectar outliers
- Validar clusters visualmente
- Comunicar resultados al negocio

### 1.4 Costo computacional y ruido

- **Tiempo de entrenamiento:** Crece con $p$ (más features = más cálculos)
- **Memoria:** Matrices más grandes
- **Ruido:** Features irrelevantes añaden ruido que confunde al modelo

### Solución: Reducción de dimensionalidad

**Idea:** Encontrar un número pequeño de **nuevas variables** (combinaciones de las originales) que capturen la mayor parte de la **variación** de los datos.

**PCA (Principal Component Analysis):** La técnica más popular para reducción lineal de dimensionalidad.

## 2. Intuición geométrica de PCA

### 2.1 Idea central

Imagina una nube de puntos en 2D con forma de elipse (alargada). PCA encuentra:

- **PC1 (Primer Componente Principal):** La dirección a lo largo de la cual los datos **varían más** (eje "largo" de la elipse)
- **PC2 (Segundo Componente Principal):** La dirección **ortogonal** a PC1 que captura la **segunda mayor varianza** (eje "corto" de la elipse)

**Clave:** Si proyectamos los datos sobre PC1, perdemos muy poca información porque los puntos están "casi" sobre esa línea.

### 2.2 Dos perspectivas equivalentes

PCA puede verse de dos formas matemáticamente equivalentes:

#### Perspectiva 1: Maximizar varianza

Buscar la dirección $v_1$ tal que la proyección de los datos sobre $v_1$ tenga **máxima varianza**.

$$
v_1 = \arg\max_{\|v\|=1} \text{Var}(X v)
$$

**Intuición:** Queremos que los datos proyectados estén lo más "dispersos" posible (no colapsados en un punto).

#### Perspectiva 2: Minimizar error de reconstrucción

Buscar la dirección $v_1$ tal que la **distancia** entre los datos originales y su proyección sea **mínima**.

$$
v_1 = \arg\min_{\|v\|=1} \sum_{i=1}^{n} \| x_i - (x_i \cdot v) v \|^2
$$

**Intuición:** Queremos que la proyección "pierda" la menor cantidad de información posible.

> 💡 **Resultado:** Ambas perspectivas dan la misma solución — los autovectores de la matriz de covarianza.

### 2.3 Analogía visual

Piensa en PCA como:
- **Rotar** el sistema de coordenadas para alinear los ejes con las direcciones de máxima varianza
- **Proyectar** los datos sobre los primeros $k$ ejes (descartar los últimos $p-k$ ejes)

**Ejemplo:** Si tienes una nube de puntos inclinada en 2D:
1. PCA rota los ejes para alinearlos con la nube
2. El nuevo eje X (PC1) apunta en la dirección de máxima variación
3. El nuevo eje Y (PC2) es perpendicular a PC1
4. Si proyectas solo sobre PC1, capturas la mayor parte de la información

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Generamos datos con correlación (forma de elipse)
rng = np.random.default_rng(0)
mean = [0, 0]  # Centrados en el origen

# Matriz de covarianza: define la forma de la elipse
# [[3, 1.6],     → var(x₁)=3, var(x₂)=1, cov(x₁,x₂)=1.6
#  [1.6, 1]]
# La covarianza positiva (1.6) hace que los datos estén correlacionados
cov = [[3, 1.6], [1.6, 1]]

# Generamos 400 puntos de una distribución normal multivariada
X = rng.multivariate_normal(mean, cov, size=400)

# Aplicamos PCA con 2 componentes (para visualizar ambos)
pca = PCA(n_components=2).fit(X)

# Extraemos los componentes principales (direcciones)
# comps: matriz (2 × 2) donde cada fila es un componente principal
# comps[0]: dirección de PC1 (vector unitario)
# comps[1]: dirección de PC2 (vector unitario, ortogonal a PC1)
comps = pca.components_

# Extraemos las varianzas explicadas (autovalores)
# expl[0]: varianza capturada por PC1 (la mayor)
# expl[1]: varianza capturada por PC2 (la segunda mayor)
expl = pca.explained_variance_

# --- Visualización ---
fig, ax = plt.subplots(figsize=(6, 6))

# Scatter plot de los datos originales
ax.scatter(X[:, 0], X[:, 1], alpha=0.4, s=15, color='gray')

# Dibujamos los componentes principales como flechas rojas
# La longitud de la flecha es proporcional a la varianza explicada
for i, (vec, var) in enumerate(zip(comps, expl)):
    # Escalamos el vector por √varianza × 2 para visualización
    v = vec * np.sqrt(var) * 2
    
    # annotate dibuja una flecha desde (0,0) hasta v
    ax.annotate('', xy=v, xytext=(0, 0),
                arrowprops={'arrowstyle': '->', 'lw': 3, 'color': 'red'})
    
    # Etiqueta del componente
    ax.text(v[0]*1.1, v[1]*1.1, f'PC{i+1}', fontsize=14,
            color='red', fontweight='bold')

ax.set_aspect('equal')  # Aspecto 1:1 para no distorsionar
ax.grid(True, alpha=0.3)
ax.set_xlabel('x₁ (original)')
ax.set_ylabel('x₂ (original)')
ax.set_title('PCA: las flechas son los ejes de máxima varianza')
plt.show()

# Mostramos la varianza explicada por cada componente
print(f'Varianza explicada por PC1: {pca.explained_variance_ratio_[0]:.1%}')
print(f'Varianza explicada por PC2: {pca.explained_variance_ratio_[1]:.1%}')

# OBSERVACIONES:
# - PC1 (flecha roja más larga): dirección de MÁXIMA varianza
#   Apunta en la dirección donde los datos están más "estirados"
# - PC2 (flecha roja más corta): dirección ORTOGONAL con segunda mayor varianza
#   Perpendicular a PC1 (90°)
# - Si proyectamos solo sobre PC1, capturamos ~75-80% de la varianza total
# - Los datos están correlacionados (elipse inclinada) → PCA los "desrota"


## 3. Formulación matemática completa

### 3.1 Datos y preprocesamiento

**Entrada:** Matriz de datos $X \in \mathbb{R}^{n \times p}$
- $n$: número de observaciones
- $p$: número de features originales

**Paso 0: Centrar los datos** (restar la media de cada columna)

$$
\tilde{X} = X - \bar{X}
$$

donde $\bar{X}$ es el vector de medias por columna.

> ⚠️ **Importante:** PCA asume datos centrados. scikit-learn lo hace automáticamente.

### 3.2 Matriz de covarianza

La matriz de covarianza mide cómo varían las features entre sí:

$$
\Sigma = \frac{1}{n-1} \tilde{X}^\top \tilde{X} \in \mathbb{R}^{p \times p}
$$

**Elementos:**
- Diagonal: $\Sigma_{ii} = \text{Var}(x_i)$ (varianza de la feature $i$)
- Fuera de diagonal: $\Sigma_{ij} = \text{Cov}(x_i, x_j)$ (covarianza entre features $i$ y $j$)

**Propiedades:**
- Simétrica: $\Sigma = \Sigma^\top$
- Semi-definida positiva: todos los autovalores $\lambda_k \ge 0$

### 3.3 Descomposición espectral (eigendecomposition)

Los **componentes principales** son los **autovectores** de $\Sigma$, ordenados por sus **autovalores**:

$$
\Sigma v_k = \lambda_k v_k \quad \text{para } k = 1, \dots, p
$$

donde:
- $v_k$: $k$-ésimo autovector (dirección del $k$-ésimo componente principal)
- $\lambda_k$: $k$-ésimo autovalor (varianza explicada por ese componente)

**Ordenamiento:** $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_p \ge 0$

**Propiedades de los autovectores:**
- **Ortogonales:** $v_i \cdot v_j = 0$ para $i \neq j$
- **Unitarios:** $\|v_k\| = 1$

### 3.4 Proyección a espacio reducido

Para reducir de $p$ a $k$ dimensiones ($k < p$):

1. **Seleccionar** los $k$ primeros autovectores: $V_k = [v_1, v_2, \dots, v_k] \in \mathbb{R}^{p \times k}$

2. **Proyectar** los datos sobre esos autovectores:

$$
Z = \tilde{X} V_k \in \mathbb{R}^{n \times k}
$$

donde $Z$ son las **coordenadas en el espacio PCA** (scores).

**Interpretación:**
- Cada fila de $Z$ es una observación en el nuevo espacio de $k$ dimensiones
- Cada columna de $Z$ es un componente principal (combinación lineal de las features originales)

### 3.5 Reconstrucción (opcional)

Podemos reconstruir una aproximación de los datos originales:

$$
\hat{X} = Z V_k^\top + \bar{X}
$$

**Error de reconstrucción:**

$$
\text{Error} = \| X - \hat{X} \|_F^2 = \sum_{k+1}^{p} \lambda_k
$$

Es decir, el error es la suma de las varianzas de los componentes **descartados**.

### 3.6 Implementación práctica: SVD

En la práctica, scikit-learn usa **Singular Value Decomposition (SVD)** en lugar de calcular $\Sigma$ explícitamente:

$$
\tilde{X} = U \Lambda V^\top
$$

donde:
- $V$: autovectores de $\Sigma$ (componentes principales)
- $\Lambda^2 / (n-1)$: autovalores de $\Sigma$ (varianzas explicadas)

**Ventajas de SVD:**
- Más estable numéricamente
- Más eficiente para $n \gg p$ o $p \gg n$
- No necesita calcular $\Sigma$ explícitamente (ahorra memoria)

## 4. Varianza explicada — ¿cuántos componentes elegir?

Cada componente captura una fracción de la **varianza total** de los datos.

### 4.1 Varianza explicada por cada componente

$$
\text{ratio}_k = \frac{\lambda_k}{\sum_{j=1}^{p} \lambda_j}
$$

**Interpretación:** Proporción de la varianza total que captura el componente $k$.

**Ejemplo:** Si $\text{ratio}_1 = 0.60$, el PC1 captura el 60% de la varianza total.

### 4.2 Varianza acumulada

$$
\text{acum}_k = \sum_{j=1}^{k} \text{ratio}_j
$$

**Interpretación:** Proporción de la varianza total capturada por los primeros $k$ componentes.

**Ejemplo:** Si $\text{acum}_3 = 0.85$, los primeros 3 componentes capturan el 85% de la varianza.

### 4.3 Heurísticas para elegir $k$

No hay una respuesta única — depende del objetivo y del problema.

#### Opción 1: Umbral de varianza acumulada

**Regla:** Elegir el $k$ más pequeño tal que $\text{acum}_k \ge \theta$ (e.g., $\theta = 0.80$ o $0.95$).

**Ventajas:**
- ✅ Simple e intuitivo
- ✅ Garantiza retener cierto % de información

**Limitaciones:**
- ❌ El umbral es arbitrario (¿80%? ¿90%? ¿95%?)
- ❌ No considera el objetivo final (clustering, visualización, etc.)

#### Opción 2: Regla de Kaiser

**Regla:** Conservar solo componentes con $\lambda_k > 1$ (cuando los datos están **estandarizados**).

**Intuición:** Un componente con $\lambda_k > 1$ explica más varianza que una variable promedio.

**Ventajas:**
- ✅ No requiere elegir umbral
- ✅ Tiene interpretación estadística

**Limitaciones:**
- ❌ Solo aplica con datos estandarizados
- ❌ Puede ser conservador (elegir pocos componentes)

#### Opción 3: Scree plot (método del codo)

**Regla:** Graficar $\lambda_k$ vs $k$ y buscar el "codo" donde la curva se aplana.

**Ventajas:**
- ✅ Visual e intuitivo
- ✅ Detecta cuándo los componentes adicionales aportan poco

**Limitaciones:**
- ❌ Subjetivo (el codo no siempre es obvio)
- ❌ Puede haber múltiples codos

#### Opción 4: Validación cruzada (para predicción)

Si PCA es preprocesamiento para un modelo supervisado:

**Regla:** Elegir el $k$ que **maximiza el rendimiento** del modelo downstream (e.g., accuracy, F1).

**Ventajas:**
- ✅ Optimiza para el objetivo final
- ✅ Basado en datos, no en heurísticas

**Limitaciones:**
- ❌ Más costoso (requiere entrenar múltiples modelos)
- ❌ Solo aplica si hay un modelo downstream

#### Opción 5: Objetivo específico

- **Visualización:** $k=2$ o $k=3$ (para graficar)
- **Compresión:** Elegir $k$ según el trade-off espacio/calidad
- **Clustering:** Probar varios $k$ y ver cuál da mejor silhouette

### 4.4 Recomendación práctica

1. **Graficar** scree plot y varianza acumulada
2. **Identificar** el codo y el umbral de 80-90%
3. **Probar** varios valores de $k$ alrededor de esos puntos
4. **Validar** con el objetivo final (clustering, predicción, etc.)

> 💡 **Regla de oro:** Empieza con $k$ que capture 80-90% de la varianza, luego ajusta según el problema.

## 5. Pre-requisitos importantes

### 5.1 Centrado y escalado (CRÍTICO)

**Centrado:** PCA asume datos centrados (media = 0). scikit-learn lo hace automáticamente.

**Escalado:** Si las variables tienen escalas muy diferentes, PCA se enfocará en las de mayor magnitud (no en las de mayor _información_).

**Ejemplo del problema:**
- `TotalCharges`: rango [18, 8684] → varianza ≈ 1,000,000
- `tenure`: rango [0, 72] → varianza ≈ 500
- Sin escalar, PC1 será casi 100% `TotalCharges` (domina por magnitud, no por información)

**Solución:** `StandardScaler` antes de PCA (media=0, std=1 para cada variable).

> ⚠️ **Regla de oro:** Siempre escalar antes de PCA, salvo que todas las variables estén en la misma escala y unidades.

### 5.2 Linealidad

PCA solo encuentra estructuras **lineales** (combinaciones lineales de las features).

**Limitación:** Si la estructura es no lineal (e.g., datos en forma de espiral), PCA no la capturará bien.

**Alternativas para no linealidad:**
- **Kernel PCA:** Aplica PCA en un espacio transformado (kernel trick)
- **t-SNE:** Reducción no lineal, ideal para visualización
- **UMAP:** Similar a t-SNE pero más rápido y preserva estructura global
- **Autoencoders:** Redes neuronales para reducción no lineal

### 5.3 Interpretabilidad

Los componentes principales son **combinaciones lineales** de todas las variables originales:

$$
PC_k = w_{k1} x_1 + w_{k2} x_2 + \dots + w_{kp} x_p
$$

**Ventaja:** Matemáticamente bien definido.

**Desventaja:** Menos interpretable que las variables originales.

**Ejemplo:** "PC1 = 0.5·tenure + 0.4·MonthlyCharges + 0.3·TotalCharges" es menos intuitivo que "tenure" solo.

**Solución:** Analizar los **loadings** (pesos $w_{kj}$) para entender qué representa cada PC.

### 5.4 Sensibilidad a outliers

PCA usa la matriz de covarianza, que es sensible a outliers (valores extremos inflan la varianza).

**Problema:** Un outlier puede "jalar" un componente principal hacia él.

**Soluciones:**
- Detectar y remover outliers antes de PCA
- Usar **Robust PCA** (variantes robustas a outliers)
- Usar **Sparse PCA** (componentes con pocos pesos no-cero)

### 5.5 Cuándo NO usar PCA

- **Variables ya son interpretables** y el negocio quiere razonar sobre ellas
- **Estructura es no lineal** → usar t-SNE, UMAP, Kernel PCA
- **$p$ es muy pequeño** (e.g., 3-5 variables) → no hay nada que reducir
- **Outliers dominan** → limpiar datos primero o usar Robust PCA
- **Variables categóricas** → PCA es para variables continuas (usar MCA, Multiple Correspondence Analysis)

## 6. Caso práctico — PCA sobre Telco Customer Churn

Vamos a:
1. Aplicar PCA al dataset de Telco con muchas variables (numéricas + dummies).
2. Mirar cuántos componentes acumulan ≥80% de la varianza.
3. Inspeccionar qué variables pesan en cada componente (loadings).
4. Combinar **K-Means + PCA** para visualizar los segmentos en 2D — el caso más útil de PCA en clustering.

In [ ]:
from pathlib import Path
import pandas as pd

# Definimos la ruta al CSV usando pathlib
DATA = Path('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Verificamos que el archivo existe
if not DATA.exists():
    raise FileNotFoundError(
        f'No se encontró {DATA}. Descarga el dataset desde '
        'https://www.kaggle.com/datasets/blastchar/telco-customer-churn '
        'y colócalo en data/ (ver README.md).'
    )

# Cargamos el CSV completo
df = pd.read_csv(DATA)

# Limpieza: TotalCharges viene como string con espacios
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)

print('Shape:', df.shape)
print(f'Clientes después de limpiar: {len(df)}')

# Mostramos las primeras filas
df.head()

# CONTEXTO:
# - 7032 clientes × 21 columnas
# - Variables demográficas: gender, SeniorCitizen, Partner, Dependents
# - Servicios: PhoneService, InternetService, StreamingTV, etc.
# - Contrato: Contract, PaymentMethod, PaperlessBilling
# - Facturación: MonthlyCharges, TotalCharges, tenure
# - Target (NO usaremos para PCA): Churn
#
# OBJETIVO:
# - Aplicar PCA para reducir dimensión
# - Visualizar clusters en 2D
# - Entender qué variables contribuyen a cada componente


In [ ]:
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Configuramos el estilo visual
sns.set_theme(style='whitegrid')

# ─────────────────────────────────────────────────────────────────
# Selección de features
# ─────────────────────────────────────────────────────────────────
# Usamos 4 numéricas + 8 categóricas (codificadas con one-hot)
# Esto nos dará ~20-30 features después del encoding

# Variables numéricas
num_cols = ['tenure',          # meses como cliente
            'MonthlyCharges',  # cuota mensual
            'TotalCharges',    # facturación acumulada
            'SeniorCitizen']   # 0/1 (ya es binaria)

# Variables categóricas (las más relevantes)
cat_cols = ['gender',           # Male / Female
            'Partner',          # Yes / No
            'Dependents',       # Yes / No
            'PhoneService',     # Yes / No
            'InternetService',  # DSL / Fiber optic / No
            'Contract',         # Month-to-month / One year / Two year
            'PaperlessBilling', # Yes / No
            'PaymentMethod']    # Electronic check / Mailed check / Bank transfer / Credit card

# ─────────────────────────────────────────────────────────────────
# One-Hot Encoding de categóricas
# ─────────────────────────────────────────────────────────────────
# get_dummies convierte cada categoría en una columna binaria
# drop_first=True elimina la primera categoría de cada variable
# (evita multicolinealidad perfecta, aunque PCA puede manejarla)

X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, drop_first=True)

# Convertimos todo a float (PCA requiere datos numéricos)
X = X.astype(float)

print('Dimensión original:', X.shape)
print(f'Features: {X.shape[1]} (4 numéricas + {X.shape[1]-4} dummies)')

# ─────────────────────────────────────────────────────────────────
# Estandarización (CRÍTICO para PCA)
# ─────────────────────────────────────────────────────────────────
# StandardScaler transforma cada columna a media=0, std=1
# Esto es ESENCIAL porque:
# 1. TotalCharges tiene rango [18, 8684] → dominaría sin escalar
# 2. tenure tiene rango [0, 72] → sería casi ignorado
# 3. Las dummies tienen rango [0, 1] → escala diferente

scaler = StandardScaler().fit(X)
X_s = scaler.transform(X)

print('\nDatos estandarizados: media≈0, std≈1')

# RESULTADO:
# - X_s tiene ~25 columnas (4 numéricas + ~21 dummies)
# - Todas las columnas tienen la misma escala
# - Ahora PCA puede comparar varianzas de forma justa


In [ ]:
from sklearn.decomposition import PCA

# Ajustamos PCA con TODOS los componentes posibles para inspeccionar la varianza
# n_components=None → calcula todos los p componentes
pca_full = PCA().fit(X_s)

# Extraemos la varianza explicada por cada componente
# ratio: array de longitud p con la proporción de varianza de cada PC
ratio = pca_full.explained_variance_ratio_

# Calculamos la varianza acumulada
# acum[k] = suma de ratio[0] + ratio[1] + ... + ratio[k]
acum = np.cumsum(ratio)

# --- Visualización ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gráfico 1: Scree plot (varianza por componente)
axes[0].bar(range(1, len(ratio) + 1), ratio, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Componente')
axes[0].set_ylabel('Varianza explicada')
axes[0].set_title('Scree Plot — Varianza por componente')
axes[0].grid(True, alpha=0.3, axis='y')

# Gráfico 2: Varianza acumulada
axes[1].plot(range(1, len(acum) + 1), acum, marker='o', linewidth=2, markersize=6)

# Líneas de referencia para 80% y 90%
axes[1].axhline(0.8, color='red', linestyle='--', linewidth=2, label='80%')
axes[1].axhline(0.9, color='green', linestyle='--', linewidth=2, label='90%')

axes[1].set_xlabel('Componente')
axes[1].set_ylabel('Varianza acumulada')
axes[1].set_title('Varianza Acumulada')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Encontramos cuántos componentes necesitamos para 80% y 90%
# searchsorted encuentra el índice donde acum >= umbral
k80 = int(np.searchsorted(acum, 0.80) + 1)
k90 = int(np.searchsorted(acum, 0.90) + 1)

print(f'Componentes para ≥80% varianza: {k80} (de {len(ratio)} totales)')
print(f'Componentes para ≥90% varianza: {k90} (de {len(ratio)} totales)')
print(f'\nReducción de dimensión:')
print(f'  Original: {X.shape[1]} features')
print(f'  Con 80%:  {k80} componentes → reducción de {(1 - k80/X.shape[1])*100:.1f}%')
print(f'  Con 90%:  {k90} componentes → reducción de {(1 - k90/X.shape[1])*100:.1f}%')

# INTERPRETACIÓN:
# - Scree plot (izquierda): Los primeros componentes capturan mucha varianza,
#   luego se aplana (codo) → componentes adicionales aportan poco
# - Varianza acumulada (derecha): Muestra cuántos componentes necesitas
#   para retener cierto % de información
#
# DECISIÓN:
# - Si priorizas compresión: usa k80 (menos componentes, pierdes 20% info)
# - Si priorizas precisión: usa k90 (más componentes, pierdes solo 10% info)
# - Para visualización: usa k=2 o k=3 (aunque pierdas más información)


In [ ]:
# --- Loadings: qué variables pesan en cada componente ---

# Mostramos los primeros 4 componentes
n_show = 4

# pca_full.components_: matriz (p × p) donde cada fila es un componente
# Extraemos las primeras n_show filas y transponemos
# Resultado: matriz (n_features × n_show) con los pesos (loadings)
loadings = pd.DataFrame(
    pca_full.components_[:n_show].T,  # Transponemos para tener features en filas
    index=X.columns,                   # Nombres de las features
    columns=[f'PC{i+1}' for i in range(n_show)],  # Nombres de los componentes
)

# --- Visualización con heatmap ---
fig, ax = plt.subplots(figsize=(8, 6))

# Heatmap: colores indican la magnitud y signo del loading
# annot=True: muestra los valores numéricos
# fmt='.2f': formato con 2 decimales
# center=0: el color blanco está en 0 (negativo=rojo, positivo=azul)
# cmap='RdBu_r': paleta rojo-blanco-azul (reversed)
sns.heatmap(loadings, annot=True, fmt='.2f', center=0,
            cmap='RdBu_r', cbar_kws={'label': 'loading'}, ax=ax)

ax.set_title(f'Loadings de los primeros {n_show} componentes')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Feature')

plt.tight_layout()
plt.show()

# CÓMO LEER LOS LOADINGS:
# 
# Cada celda muestra el peso (coeficiente) de una feature en un componente:
# PC_k = w_{k,1}·feature_1 + w_{k,2}·feature_2 + ... + w_{k,p}·feature_p
#
# - **Valor positivo grande (azul oscuro):** La feature empuja la observación
#   hacia el lado positivo del componente
# - **Valor negativo grande (rojo oscuro):** Empuja hacia el lado negativo
# - **Valor cercano a 0 (blanco):** La feature NO contribuye a ese componente
#
# EJEMPLO DE INTERPRETACIÓN (adaptar a tus resultados):
# - Si PC1 tiene loadings altos en tenure, MonthlyCharges, TotalCharges
#   → PC1 representa "antigüedad y gasto" (clientes leales de alto valor)
# - Si PC2 tiene loading alto en Contract_Month-to-month (positivo)
#   y Contract_Two year (negativo)
#   → PC2 representa "tipo de contrato" (corto plazo vs largo plazo)
#
# UTILIDAD:
# - Entender qué significa cada componente en términos de las variables originales
# - Identificar qué variables son más importantes en la estructura de los datos
# - Dar nombres interpretables a los componentes (e.g., "PC1 = Valor del cliente")


**Cómo leer los loadings:**
- Un valor positivo grande (azul) → la variable empuja la observación hacia el lado positivo del componente.
- Un valor negativo grande (rojo) → empuja al lado negativo.
- Valores cercanos a 0 → esa variable no contribuye al componente.

Esto nos permite ponerle un nombre a cada PC (p. ej., "PC1 = antigüedad y gasto acumulado").

## 7. Aplicación: K-Means + PCA para visualizar clusters

Esta es **la combinación más usada** de PCA en la práctica del clustering:

### ¿Por qué combinar K-Means y PCA?

**Problema:** Con $p > 3$ features, no podemos visualizar los clusters directamente.

**Solución:** Proyectar los datos a 2D con PCA y graficar los clusters.

### Dos estrategias

| Estrategia | Descripción | Cuándo usar |
|------------|-------------|-------------|
| **1. Clusterizar en alta dimensión, visualizar con PCA** | Entrenar K-Means sobre $X$ escalado (todas las features). Proyectar las etiquetas sobre PC1-PC2 para visualizar. | Cuando quieres usar toda la información para clustering |
| **2. Reducir primero con PCA, clusterizar en espacio reducido** | Aplicar PCA a $k$ componentes. Entrenar K-Means sobre $Z$ (espacio PCA). | Cuando hay mucho ruido o redundancia. Más rápido. |

### Ventajas y limitaciones

**Estrategia 1 (clusterizar en alta dim):**
- ✅ Usa toda la información disponible
- ✅ No pierde información por reducción prematura
- ❌ Más lento (más features)
- ❌ Puede sufrir de maldición de la dimensión

**Estrategia 2 (reducir primero):**
- ✅ Más rápido (menos features)
- ✅ Elimina ruido y redundancia
- ✅ Evita maldición de la dimensión
- ❌ Pierde información (según cuántos componentes uses)
- ❌ Los clusters pueden ser diferentes

### ¿Cuál elegir?

- Si $p$ es moderado (< 50) y los datos son limpios → **Estrategia 1**
- Si $p$ es grande (> 100) o hay mucho ruido → **Estrategia 2**
- Si no estás seguro → **Prueba ambas** y compara silhouette

> 💡 **Importante:** La visualización en 2D puede ser engañosa. Si PC1-PC2 solo capturan 30% de la varianza, los clusters pueden verse mezclados aunque estén bien separados en alta dimensión.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Elegimos K=4 para este ejemplo (en la práctica, usarías codo/silhouette)
K = 4

# ─────────────────────────────────────────────────────────────────
# ESTRATEGIA 1: Clusterizar en alta dimensión, visualizar con PCA(2)
# ─────────────────────────────────────────────────────────────────

# Entrenamos K-Means sobre TODOS los datos escalados (alta dimensión)
# n_init=20: 20 inicializaciones para asegurar buen mínimo
km_full = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_s)

# Calculamos silhouette en el espacio original (alta dimensión)
sil_full = silhouette_score(X_s, km_full.labels_)

# Para visualizar, proyectamos a 2D con PCA
# n_components=2: solo los 2 primeros componentes
pca2 = PCA(n_components=2).fit(X_s)
Z2 = pca2.transform(X_s)  # Proyección de los datos

# ─────────────────────────────────────────────────────────────────
# ESTRATEGIA 2: PCA primero, K-Means después
# ─────────────────────────────────────────────────────────────────

# Aplicamos PCA con k80 componentes (≥80% varianza)
pcaK = PCA(n_components=k80).fit(X_s)
Z_red = pcaK.transform(X_s)  # Datos en espacio reducido

# Entrenamos K-Means en el espacio PCA reducido
km_red = KMeans(n_clusters=K, n_init=20, random_state=42).fit(Z_red)

# Calculamos silhouette en el espacio reducido
sil_red = silhouette_score(Z_red, km_red.labels_)

# ─────────────────────────────────────────────────────────────────
# Comparación de métricas
# ─────────────────────────────────────────────────────────────────
print('COMPARACIÓN DE ESTRATEGIAS:')
print(f'Estrategia 1 (clusterizar en alta dim):')
print(f'  - Dimensión: {X_s.shape[1]} features')
print(f'  - Silhouette: {sil_full:.3f}')
print(f'\nEstrategia 2 (PCA primero):')
print(f'  - Dimensión: {k80} componentes ({k80/X_s.shape[1]*100:.1f}% de features)')
print(f'  - Varianza retenida: {acum[k80-1]:.1%}')
print(f'  - Silhouette: {sil_red:.3f}')

# INTERPRETACIÓN:
# - Si sil_full > sil_red: Clusterizar en alta dimensión es mejor
#   (la información perdida por PCA era relevante para los clusters)
# - Si sil_red > sil_full: Reducir primero es mejor
#   (PCA eliminó ruido que confundía a K-Means)
# - Si son similares: Ambas estrategias son equivalentes
#   (usa la estrategia 2 por eficiencia)
#
# NOTA: La diferencia suele ser pequeña (< 0.05) en la práctica


In [ ]:
# --- Visualización 2D de ambas estrategias ---

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ─────────────────────────────────────────────────────────────────
# Gráfico 1: Estrategia 1 (clusterizar en alta dim)
# ─────────────────────────────────────────────────────────────────

# Reproyectamos los centroides (que están en alta dim) a PC1-PC2
# km_full.cluster_centers_: matriz (K × p) con los centroides en alta dim
# pca2.transform: proyecta a 2D
centros2_full = pca2.transform(km_full.cluster_centers_)

# Scatter plot de los datos proyectados, coloreados por cluster
axes[0].scatter(Z2[:, 0], Z2[:, 1], c=km_full.labels_,
                cmap='tab10', s=10, alpha=0.6)

# Centroides proyectados (X negras)
axes[0].scatter(centros2_full[:, 0], centros2_full[:, 1],
                marker='X', c='black', s=200, edgecolor='white',
                linewidth=2, label='Centroides')

# Etiquetas de los ejes con % de varianza explicada
axes[0].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title(f'Estrategia 1: K-Means en alta dim → proyectado en PCA(2)\nSilhouette = {sil_full:.3f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ─────────────────────────────────────────────────────────────────
# Gráfico 2: Estrategia 2 (PCA primero)
# ─────────────────────────────────────────────────────────────────

# Los centroides ya viven en el espacio PCA reducido
# Extraemos solo las primeras 2 dimensiones para visualizar
centros2_red = km_red.cluster_centers_[:, :2]

# Scatter plot de los datos en espacio PCA reducido
axes[1].scatter(Z_red[:, 0], Z_red[:, 1], c=km_red.labels_,
                cmap='tab10', s=10, alpha=0.6)

# Centroides en espacio PCA
axes[1].scatter(centros2_red[:, 0], centros2_red[:, 1],
                marker='X', c='black', s=200, edgecolor='white',
                linewidth=2, label='Centroides')

axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title(f'Estrategia 2: PCA({k80}) → K-Means\nSilhouette = {sil_red:.3f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# CÓMO INTERPRETAR LA VISUALIZACIÓN:
#
# 1. **Separación de colores:**
#    - Si los colores están bien separados → clusters robustos
#    - Si los colores se entremezclan → clusters débiles o PC1-PC2 no capturan
#      suficiente información
#
# 2. **Posición de centroides:**
#    - Deben estar en el "centro de masa" de cada color
#    - Si un centroide está entre dos colores → posible mal clustering
#
# 3. **Varianza explicada:**
#    - Si PC1+PC2 capturan < 40% de varianza → la visualización puede ser engañosa
#    - Los clusters pueden estar bien separados en alta dimensión pero verse
#      mezclados en 2D
#
# 4. **Comparación entre estrategias:**
#    - Si las visualizaciones son similares → ambas estrategias encuentran
#      la misma estructura
#    - Si son muy diferentes → la reducción de dimensión afecta el clustering
#
# ADVERTENCIA: Esta visualización es solo una PROYECCIÓN de un espacio
# de alta dimensión. No confíes solo en ella — valida con silhouette,
# análisis de centroides y validación de negocio.


**Lectura de la visualización:**
- Si en el scatter PC1×PC2 vemos **manchas separadas** del color de cada cluster, la estructura es clara y robusta.
- Si los colores se entremezclan, el clustering puede no ser el adecuado para los datos — o los 2 primeros PCs no capturan suficiente varianza para ser representativos.
- **Cuidado**: 2 componentes pueden explicar solo, p. ej., 30% de la varianza. Que se vean mezclados en PC1×PC2 no significa necesariamente que los clusters estén mal.

In [ ]:
# --- Validación externa con Churn ---

# Asignamos los clusters de la estrategia 2 (PCA primero) al DataFrame
df['cluster_pca'] = km_red.labels_

# Convertimos Churn a binario
df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)

# Calculamos estadísticas por cluster
resumen = df.groupby('cluster_pca').agg(
    n_clientes=('Churn_bin', 'size'),              # tamaño del cluster
    tasa_churn=('Churn_bin', 'mean'),              # proporción de churn
    tenure_medio=('tenure', 'mean'),                # antigüedad promedio
    cargo_mensual_medio=('MonthlyCharges', 'mean'), # gasto mensual promedio
).round(3)

print('Resumen de clusters (con validación externa):')
resumen

# INTERPRETACIÓN CLAVE:
#
# Si la tasa de churn VARÍA entre clusters, los segmentos descubiertos
# capturan algo REAL del negocio, aunque NUNCA usamos Churn para entrenar.
#
# EJEMPLO DE INTERPRETACIÓN (adaptar a tus resultados):
# - Cluster 0: tenure bajo, charges bajos, churn alto (40%)
#   → "Clientes nuevos en riesgo"
# - Cluster 1: tenure alto, charges altos, churn bajo (10%)
#   → "Clientes leales de alto valor"
# - Cluster 2: tenure bajo, charges altos, churn medio (25%)
#   → "Clientes premium recientes"
# - Cluster 3: tenure medio, charges medios, churn medio (20%)
#   → "Clientes estándar"
#
# ACCIÓN DE NEGOCIO:
# - Cluster de alto churn → campaña de retención
# - Cluster de bajo churn → programa de fidelización, upselling
# - Cluster premium → atención personalizada, evitar que se vayan
#
# VALIDACIÓN:
# Si los clusters NO se diferencian en churn (tasas similares),
# puede significar:
# 1. Las variables usadas no predicen churn
# 2. Necesitas más/mejores features
# 3. El K elegido no es apropiado


## 8. Cuándo NO usar PCA

- Cuando las variables originales **ya son interpretables** y el negocio quiere razonar sobre ellas (PCA crea ejes abstractos).
- Cuando la estructura es **no lineal** — t-SNE / UMAP / Kernel PCA suelen visualizarla mejor.
- Cuando $p$ es **muy pequeño** (p. ej., 3–5 variables): no hay nada que reducir.
- Cuando los **outliers** dominan: la varianza está inflada por unos pocos puntos extremos.

## 9. Referencias

- Pearson, K. (1901). *On lines and planes of closest fit to systems of points in space*.
- Hotelling, H. (1933). *Analysis of a complex of statistical variables into principal components*.
- ISLR cap. 12.2 — *Principal Components Analysis*.
- scikit-learn: https://scikit-learn.org/stable/modules/decomposition.html#pca
- Jolliffe, I. T. (2002). *Principal Component Analysis*.

## Predicción sobre datos nuevos — uso del pipeline en producción

PCA es **transformador**, no predictor — pero es una pieza fundamental del pipeline. La regla de oro: el `scaler`, el `pca` y el `kmeans` deben persistirse **juntos** y aplicarse en el mismo orden a los datos nuevos.

In [ ]:
import joblib

# ─────────────────────────────────────────────────────────────────
# PASO 1: Reentrenamos todo el pipeline con TODOS los datos
# ─────────────────────────────────────────────────────────────────
# Ya validamos K y el número de componentes. Ahora usamos el 100% de los datos.

# Reajustamos el scaler con todos los datos
scaler_final = StandardScaler().fit(X)
X_s_final = scaler_final.transform(X)

# Reajustamos PCA con k80 componentes
pca_final = PCA(n_components=k80).fit(X_s_final)

# Entrenamos K-Means final en el espacio PCA
km_final = KMeans(n_clusters=K, n_init=20, random_state=42).fit(
    pca_final.transform(X_s_final)
)

# ─────────────────────────────────────────────────────────────────
# PASO 2: Creamos un bundle con TODO el pipeline
# ─────────────────────────────────────────────────────────────────
# CRÍTICO: Necesitamos guardar TODA la cadena de transformaciones

bundle = {
    'scaler':   scaler_final,      # StandardScaler (media y std de cada feature)
    'pca':      pca_final,          # PCA (autovectores y autovalores)
    'kmeans':   km_final,           # K-Means (centroides en espacio PCA)
    'columnas': X.columns.tolist(), # Orden exacto de columnas (post one-hot)
    'num_cols': num_cols,           # Columnas numéricas originales
    'cat_cols': cat_cols,           # Columnas categóricas originales
    'K':        K,                  # Número de clusters
    'k_pca':    k80,                # Número de componentes PCA
}

# ─────────────────────────────────────────────────────────────────
# PASO 3: Persistimos el bundle a disco
# ─────────────────────────────────────────────────────────────────
joblib.dump(bundle, 'pipeline_telco_pca_kmeans.pkl')
print('Pipeline guardado: pipeline_telco_pca_kmeans.pkl')

# Para recuperarlo en otro proceso:
# bundle = joblib.load('pipeline_telco_pca_kmeans.pkl')
# scaler = bundle['scaler']
# pca = bundle['pca']
# kmeans = bundle['kmeans']
# columnas = bundle['columnas']

# IMPORTANTE: En producción real también deberías guardar:
# - Fecha de entrenamiento y versión del pipeline
# - Métricas de validación (silhouette, varianza explicada)
# - Descripción de cada cluster (nombres, características)
# - Loadings de PCA (para interpretar componentes)
# - Distribuciones de las features (para detectar data drift)
#
# ORDEN DE APLICACIÓN (CRÍTICO):
# Para un nuevo cliente, SIEMPRE aplicar en este orden:
# 1. One-hot encoding (mismo drop_first=True)
# 2. Alinear columnas con X.columns
# 3. scaler.transform()
# 4. pca.transform()
# 5. kmeans.predict()
#
# Si cambias el orden o te saltas un paso, las predicciones serán incorrectas.


### Inferencia individual — un cliente nuevo pasando por todo el pipeline

In [ ]:
# Definimos un cliente hipotético
# En producción esto vendría de una API, base de datos, CRM, etc.
nuevo_cliente_raw = pd.DataFrame([{
    # Numéricas
    'tenure':           5,              # 5 meses como cliente (nuevo)
    'MonthlyCharges':   85.0,           # $85/mes (medio-alto)
    'TotalCharges':     425.0,          # $425 total (5 × $85)
    'SeniorCitizen':    0,              # No es adulto mayor
    
    # Categóricas
    'gender':           'Female',
    'Partner':          'No',
    'Dependents':       'No',
    'PhoneService':     'Yes',
    'InternetService':  'Fiber optic',  # Servicio premium
    'Contract':         'Month-to-month', # Sin compromiso (riesgo de churn)
    'PaperlessBilling': 'Yes',
    'PaymentMethod':    'Electronic check',
}])

# ─────────────────────────────────────────────────────────────────
# PASO CRÍTICO: Replicar EXACTAMENTE el preprocesamiento del entrenamiento
# ─────────────────────────────────────────────────────────────────

# 1. One-hot encoding (mismo drop_first=True)
nuevo = pd.get_dummies(nuevo_cliente_raw, columns=cat_cols, drop_first=True)

# 2. Alinear columnas con el entrenamiento
#    Si falta una columna dummy (e.g., Contract_Two year), se rellena con 0
#    Si sobra una columna, se elimina
nuevo = nuevo.reindex(columns=X.columns, fill_value=0).astype(float)

# 3. Aplicar el pipeline completo
# a) Estandarizar
nuevo_scaled = scaler_final.transform(nuevo)

# b) Proyectar a espacio PCA
z = pca_final.transform(nuevo_scaled)

# c) Asignar al cluster más cercano
cluster_id = km_final.predict(z)[0]

# ─────────────────────────────────────────────────────────────────
# Resultado
# ─────────────────────────────────────────────────────────────────
print(f'Coordenadas en PCA (primeros 3 componentes):')
print(f'  PC1 = {z[0, 0]:.2f}')
print(f'  PC2 = {z[0, 1]:.2f}')
print(f'  PC3 = {z[0, 2]:.2f}')
print(f'\nCluster asignado: {cluster_id}')
print(f'\nCaracterísticas del cluster {cluster_id}:')
print(resumen.loc[cluster_id])

# INTERPRETACIÓN:
# - El cliente tiene tenure bajo (5 meses) pero MonthlyCharges medio-alto ($85)
# - Contrato mes a mes → alto riesgo de churn
# - Probablemente caiga en un cluster de "Clientes recientes con riesgo"
#
# ACCIÓN DE NEGOCIO:
# - Ofrecerle descuento por contratar plan anual (reducir churn)
# - Programa de bienvenida personalizado
# - Monitorear satisfacción (encuestas, NPS)
# - Cross-selling de servicios adicionales
#
# VENTAJA DE PCA + K-MEANS:
# - Puedes ubicar al cliente en el mapa 2D (PC1 vs PC2) y mostrarlo
#   al equipo de negocio para explicar la segmentación
# - Los componentes principales pueden tener interpretación
#   (e.g., PC1 = "valor del cliente", PC2 = "tipo de contrato")


**Lecciones para producción:**
- El número de componentes (`k80`) es un hiperparámetro: documenta en qué porcentaje de varianza acumulada lo elegiste.
- Si las variables originales cambian (nueva campaña, nuevo producto) hay que **reentrenar PCA**, no solo el K-Means.
- Para visualizaciones rápidas en dashboards, persistir el `pca` con `n_components=2` es muy útil — permite ubicar nuevos clientes en el mismo mapa que se usó para presentar la segmentación al negocio.